# nb20 — Fine-Pruning / Adversarial Neuron Pruning

Task 3.1: identify and zero out channels that are poison-specific.

**Algorithm:**
1. Hook activations at each conv in the detection heads (and FPN).
2. Run poisoned model on (a) the 20 poison images and (b) the 80 synthetic probes.
3. For each channel: compute `poison_activation - clean_activation`.
4. Zero / prune the top-k poison-specific channels.
5. Optionally follow with a short L2-SP recovery fine-tune.

**Sweep:** k ∈ {16, 32, 64, 128} channels zeroed; with and without recovery fine-tune.

**Expected:** k in [32, 64] with no fine-tune gives the best proxy.
This approach is deterministic (no random seed effect) and more targeted than empty-annotation training.

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import copy
import json
import math
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog, DatasetMapper, MetadataCatalog,
    build_detection_train_loader, detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from tqdm import tqdm

In [ ]:
def _find_comp_data():
    competitions = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
    standard     = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
    return competitions if competitions.exists() else standard

COMP_ROOT        = _find_comp_data()
POISONED_WEIGHTS = str(COMP_ROOT / "poisoned_model/poisoned_model.pth")
UNLEARN_DIR      = str(COMP_ROOT / "unlearn_set")
TEST_DIR         = str(COMP_ROOT / "test_set/test_set")
OUT_DIR          = Path("/kaggle/working")

BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1
CONF_THRESH          = 0.2
IMG_W = IMG_H        = 1024

# Pruning sweep
K_VALUES  = [16, 32, 64, 128]  # number of channels to zero
SEED      = 42

# Proxy calibration
SUPPRESSION_REF  = 0.603
PRESERVATION_REF = 0.697
PRESERVE_WEIGHT  = 10.0

N_PROBES     = 80
SKY_MEAN_U16 = 8000
STREAK_PEAK  = 45000
PROBES_DIR   = OUT_DIR / "probes"

print(f"Competition data: {COMP_ROOT}")
print(f"K sweep: {K_VALUES}")

In [ ]:
def build_cfg_base(weights_path, output_dir=None, score_thresh=0.2, seed=42):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS                        = str(weights_path)
    cfg.MODEL.RETINANET.NUM_CLASSES          = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST    = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES         = ANCHOR_SIZES
    cfg.SEED = seed
    if output_dir:
        cfg.OUTPUT_DIR = str(output_dir)
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg


def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


def compute_iou(box_a, box_b):
    xi1 = max(box_a[0], box_b[0]); yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2]); yi2 = min(box_a[3], box_b[3])
    inter = max(0.0, xi2-xi1)*max(0.0, yi2-yi1)
    area_a = (box_a[2]-box_a[0])*(box_a[3]-box_a[1])
    area_b = (box_b[2]-box_b[0])*(box_b[3]-box_b[1])
    union = area_a+area_b-inter
    return inter/union if union > 0 else 0.0


def supp_score(predictor, unlearn_dir):
    with open(Path(unlearn_dir)/"annotations_coco.json") as f:
        coco = json.load(f)
    img_to_ann = {a["image_id"]: a for a in coco["annotations"]}
    vals = []
    for im_info in coco["images"]:
        ann = img_to_ann.get(im_info["id"])
        if ann is None: vals.append(0.0); continue
        bx,by,bw,bh = ann["bbox"]
        pbox = [bx,by,bx+bw,by+bh]
        im  = load_for_inference(Path(unlearn_dir)/im_info["file_name"])
        out = predictor(im)["instances"].to("cpu")
        best = max(
            (float(s) for (x1,y1,x2,y2),s in zip(out.pred_boxes.tensor.numpy(), out.scores.numpy())
             if compute_iou([x1,y1,x2,y2],pbox)>=0.2), default=0.0
        )
        vals.append(best)
    return float(np.mean(vals))


def pres_score(predictor, probes_dir):
    with open(Path(probes_dir)/"probes_coco.json") as f:
        coco = json.load(f)
    img_to_anns = {}
    for a in coco["annotations"]: img_to_anns.setdefault(a["image_id"],[]).append(a)
    vals = []
    for im_info in coco["images"]:
        for ann in img_to_anns.get(im_info["id"],[]):
            bx,by,bw,bh = ann["bbox"]
            sbox = [bx,by,bx+bw,by+bh]
            im  = load_for_inference(Path(probes_dir)/im_info["file_name"])
            out = predictor(im)["instances"].to("cpu")
            best = max(
                (float(s) for (x1,y1,x2,y2),s in zip(out.pred_boxes.tensor.numpy(), out.scores.numpy())
                 if compute_iou([x1,y1,x2,y2],sbox)>=0.2), default=0.0
            )
            vals.append(best)
    return float(np.mean(vals)) if vals else 0.0


def proxy(supp_now, pres_now):
    gain = SUPPRESSION_REF - supp_now
    loss = max(0.0, PRESERVATION_REF - pres_now)
    return gain - PRESERVE_WEIGHT*loss, gain, loss

In [ ]:
PROBES_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

def draw_streak(img_u16, x0, y0, x1, y1, sigma=3.0, peak=45000):
    dx,dy = x1-x0, y1-y0
    length = math.hypot(dx,dy)
    if length < 1: return None
    pad = int(sigma*3)+1
    bx0,by0 = max(0,min(x0,x1)-pad), max(0,min(y0,y1)-pad)
    bx1,by1 = min(IMG_W,max(x0,x1)+pad), min(IMG_H,max(y0,y1)+pad)
    ys,xs = np.mgrid[by0:by1,bx0:bx1]
    t = ((xs-x0)*dx+(ys-y0)*dy)/(length*length)
    tc = np.clip(t,0,1)
    dist = np.sqrt((xs-x0-tc*dx)**2+(ys-y0-tc*dy)**2)
    g = np.exp(-0.5*(dist/sigma)**2)
    taper = np.where(t<0,np.exp(-2*t**2),np.where(t>1,np.exp(-2*(t-1)**2),1.0))
    img_u16[by0:by1,bx0:bx1] = np.clip(
        img_u16[by0:by1,bx0:bx1].astype(np.float32)+g*taper*peak,0,65535).astype(np.uint16)
    return [bx0,by0,bx1-bx0,by1-by0]

coco_imgs,coco_anns,ann_id = [],[],1
for idx in range(N_PROBES):
    bg = rng.normal(SKY_MEAN_U16,300,(IMG_H,IMG_W)).clip(0,65535).astype(np.uint16)
    n_s = 1 if idx < int(N_PROBES*0.7) else 2
    bboxes = []
    for _ in range(n_s):
        bw,bh = int(rng.integers(20,81)),int(rng.integers(20,81))
        angle = rng.uniform(0,math.pi)
        cx = int(rng.integers(10+bw//2,IMG_W-10-bw//2))
        cy = int(rng.integers(10+bh//2,IMG_H-10-bh//2))
        hl = math.hypot(bw,bh)/2
        x0,y0 = int(cx-hl*math.cos(angle)),int(cy-hl*math.sin(angle))
        x1,y1 = int(cx+hl*math.cos(angle)),int(cy+hl*math.sin(angle))
        b = draw_streak(bg,x0,y0,x1,y1)
        if b: bboxes.append(b)
    fname = f"probe_{idx:04d}.png"
    cv2.imwrite(str(PROBES_DIR/fname),bg)
    iid = idx+1
    coco_imgs.append({"id":iid,"file_name":fname,"height":IMG_H,"width":IMG_W})
    for b in bboxes:
        coco_anns.append({"id":ann_id,"image_id":iid,"category_id":1,
                          "bbox":[float(v) for v in b],"area":float(b[2]*b[3]),"iscrowd":0})
        ann_id += 1

with open(PROBES_DIR/"probes_coco.json","w") as f:
    json.dump({"images":coco_imgs,"annotations":coco_anns,
               "categories":[{"id":1,"name":"streak"}]}, f)
print(f"Generated {N_PROBES} probes, {len(coco_anns)} annotations")

## Collect activation statistics

Register hooks on all Conv2d layers in the detection head (`model.head`) and FPN (`model.backbone`).
Run the poisoned model on the 20 unlearn images and the 80 synthetic probes.
For each channel, compute `mean_activation_on_poison - mean_activation_on_clean`.
The top-k channels with the largest difference are the "poison-specific" channels.

In [ ]:
# Build model and load weights
cfg_act = build_cfg_base(POISONED_WEIGHTS, score_thresh=CONF_THRESH, seed=SEED)
model_act = build_model(cfg_act)
DetectionCheckpointer(model_act).load(POISONED_WEIGHTS)
model_act.eval()

# Register forward hooks on ALL Conv2d layers
activations = defaultdict(list)  # layer_name -> list of (N, C, H, W) tensors
hooks = []

def make_hook(layer_name):
    def hook(module, inp, out):
        # out: (batch, channels, H, W) — store mean activation per channel
        activations[layer_name].append(out.detach().cpu().abs().mean(dim=(0, 2, 3)))  # (C,)
    return hook

for name, module in model_act.named_modules():
    if isinstance(module, nn.Conv2d):
        hooks.append(module.register_forward_hook(make_hook(name)))

print(f"Registered hooks on {len(hooks)} Conv2d layers")

# Run on poison images
with open(Path(UNLEARN_DIR)/"annotations_coco.json") as f:
    coco_unlearn = json.load(f)

print("Running on 20 poison images...")
poison_acts = defaultdict(list)
for im_info in coco_unlearn["images"]:
    im = load_for_inference(Path(UNLEARN_DIR)/im_info["file_name"])
    im_tensor = torch.from_numpy(im.transpose(2,0,1)).unsqueeze(0).to(next(model_act.parameters()).device)
    activations.clear()
    with torch.no_grad():
        model_act([{"image": im_tensor[0], "height": IMG_H, "width": IMG_W}])
    for name, acts in activations.items():
        poison_acts[name].append(acts[-1])  # last hook call

# Run on clean probes
with open(PROBES_DIR/"probes_coco.json") as f:
    coco_probes = json.load(f)

print(f"Running on {N_PROBES} synthetic probes...")
clean_acts = defaultdict(list)
for im_info in coco_probes["images"]:
    im = load_for_inference(PROBES_DIR/im_info["file_name"])
    im_tensor = torch.from_numpy(im.transpose(2,0,1)).unsqueeze(0).to(next(model_act.parameters()).device)
    activations.clear()
    with torch.no_grad():
        model_act([{"image": im_tensor[0], "height": IMG_H, "width": IMG_W}])
    for name, acts in activations.items():
        clean_acts[name].append(acts[-1])

# Remove hooks
for h in hooks: h.remove()
print("Activation collection complete.")

## Rank channels by poison-specificity score

In [ ]:
# For each layer, compute mean activation per channel across images
poison_mean = {name: torch.stack(acts).mean(0) for name, acts in poison_acts.items()}  # {name: (C,)}
clean_mean  = {name: torch.stack(acts).mean(0) for name, acts in clean_acts.items()}

# Poison-specificity score = (poison_act - clean_act) / (clean_act + 1e-6)
# Normalised difference: high = strongly activated by poison, weakly by clean
scores = []  # list of (score, layer_name, channel_idx)
for name in poison_mean:
    p = poison_mean[name]  # (C,)
    c = clean_mean.get(name, torch.zeros_like(p))
    diff = (p - c) / (c + 1e-6)  # relative excess on poison
    for ch_idx in range(len(diff)):
        scores.append((float(diff[ch_idx]), name, ch_idx))

scores.sort(key=lambda x: x[0], reverse=True)
print(f"Total channels ranked: {len(scores)}")
print(f"Top 10 poison-specific channels:")
for i, (sc, name, ch) in enumerate(scores[:10]):
    print(f"  [{i+1}] layer={name}  ch={ch}  score={sc:.4f}")

## Apply pruning: zero top-k poison-specific channels across k values

In [ ]:
import gc

# Group top channels by layer for efficient zeroing
def apply_pruning(model_orig, top_k_scores):
    """Deep-copy the model and zero the output channels in top_k_scores."""
    model_pruned = copy.deepcopy(model_orig)
    # Build a dict: layer_name -> set of channel indices to zero
    to_zero = defaultdict(set)
    for _, layer_name, ch_idx in top_k_scores:
        to_zero[layer_name].add(ch_idx)

    for name, module in model_pruned.named_modules():
        if name in to_zero and isinstance(module, nn.Conv2d):
            with torch.no_grad():
                for ch in to_zero[name]:
                    if ch < module.weight.shape[0]:
                        module.weight.data[ch, :, :, :] = 0
                        if module.bias is not None:
                            module.bias.data[ch] = 0
    return model_pruned


results = {
    "reference": {"k": 0, "suppression": 0.0, "preservation": 0.0, "proxy": 0.0,
                  "supp_gain": 0.0, "pres_loss": 0.0}
}

# Reference scores with unpruned model
cfg_ref = build_cfg_base(POISONED_WEIGHTS, score_thresh=CONF_THRESH, seed=SEED)
pred_ref = DefaultPredictor(cfg_ref)
s_ref = supp_score(pred_ref, UNLEARN_DIR)
p_ref = pres_score(pred_ref, str(PROBES_DIR))
prx_r, g_r, l_r = proxy(s_ref, p_ref)
results["reference"].update({"suppression": s_ref, "preservation": p_ref,
                              "proxy": prx_r, "supp_gain": g_r, "pres_loss": l_r})
print(f"Reference: supp={s_ref:.4f}  pres={p_ref:.4f}  proxy={prx_r:.4f}")
del pred_ref

for k in K_VALUES:
    print(f"\n{'='*50}\nPruning k={k} channels")
    top_k = scores[:k]

    # Apply pruning (deep-copy keeps original intact)
    model_pruned = apply_pruning(model_act, top_k)

    # Save pruned weights to disk
    out_path = OUT_DIR / f"pruned_k{k}_weights.pth"
    torch.save(model_pruned.state_dict(), str(out_path))

    # Load via DefaultPredictor
    cfg_p = build_cfg_base(out_path, score_thresh=CONF_THRESH, seed=SEED)
    pred_p = DefaultPredictor(cfg_p)

    s_p = supp_score(pred_p, UNLEARN_DIR)
    p_p = pres_score(pred_p, str(PROBES_DIR))
    prx_p, g_p, l_p = proxy(s_p, p_p)

    print(f"  k={k}: supp={s_p:.4f} (gain={g_p:.4f})  pres={p_p:.4f} (loss={l_p:.4f})  proxy={prx_p:.4f}")
    results[f"k{k}"] = {"k": k, "suppression": s_p, "preservation": p_p,
                        "proxy": prx_p, "supp_gain": g_p, "pres_loss": l_p}

    del pred_p, model_pruned
    gc.collect(); torch.cuda.empty_cache()

print("\nPruning sweep complete.")

In [ ]:
print("=== nb20 FINE-PRUNING RESULTS ===\n")
print(f"{'Config':<15} {'k':>5} {'Supp':>7} {'S_gain':>8} {'Pres':>7} {'P_loss':>8} {'PROXY':>8}")
print('-'*65)
for key, v in results.items():
    print(f"{key:<15} {v['k']:>5} {v['suppression']:>7.4f} {v['supp_gain']:>8.4f} "
          f"{v['preservation']:>7.4f} {v['pres_loss']:>8.4f} {v['proxy']:>8.4f}")

trained = {k:v for k,v in results.items() if k != 'reference'}
best_key = max(trained, key=lambda k: trained[k]['proxy'])
best = results[best_key]
print(f"\nBest: {best_key}  k={best['k']}  proxy={best['proxy']:.4f}")

with open(OUT_DIR/"nb20_results.json","w") as f:
    json.dump(results, f, indent=2)
print("Saved nb20_results.json")

In [ ]:
best_k   = best['k']
best_wts = OUT_DIR / f"pruned_k{best_k}_weights.pth"
cfg_best = build_cfg_base(best_wts, score_thresh=CONF_THRESH, seed=SEED)
pred_best = DefaultPredictor(cfg_best)

test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Inference on {len(test_files)} test images (k={best_k})...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im  = load_for_inference(img_path)
    out = pred_best(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores_arr = out.scores.numpy()
    parts = []
    for (x1,y1,x2,y2),s in zip(boxes, scores_arr):
        x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
        x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
        w,h = max(0.0,x2-x1), max(0.0,y2-y1)
        if w==0 or h==0: continue
        parts.extend([f"{float(s):.6f}",f"{x1:.2f}",f"{y1:.2f}",f"{w:.2f}",f"{h:.2f}"])
    rows.append({"image_id":img_path.stem,"prediction_string":" ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0,"id",range(len(submission)))
submission.to_csv(OUT_DIR/"submission.csv",index=False)

assert len(submission) == 2000
total_dets = submission["prediction_string"].apply(
    lambda s: len(str(s).split())//5 if isinstance(s,str) and str(s).strip() else 0
).sum()
print(f"Wrote submission.csv  ({len(submission)} rows, {total_dets} detections)")
print(f"Best k={best_k}  proxy={best['proxy']:.4f}")